# EPIC Participant SDK Quickstart

This notebook walks through the complete participant workflow: connect to an EPIC server, collect live contest observations, inspect the data, make a simple prediction, submit it, and check results. Replace the placeholder values with the contest and account details from your instructor or organizer.

## Installation — run this cell once to install the SDK and its dependencies.

In [ ]:
%pip install "epic-elios-client[notebook]"

## Configure the Client

The SDK needs the base URL of the EPIC server and your contest ID. The client stores your authentication token after login so later REST calls and WebSocket connections are authenticated automatically.

In [ ]:
from epic_client import EPICClient

SERVER_URL = "https://epic.elioslab.net"
client = EPICClient(SERVER_URL)

## 1. Log In

Log in with your EPIC username and password. The server returns a JWT access token, and the SDK saves it for future calls.

In [ ]:
client.login("student1", "correct-password")

## 2. Browse Available Contests

Use `list_contests()` to see contests available on the server. Look for contests with status `ACTIVE`, then copy the `contest_id` of the contest you want to join into the `CONTEST_ID` variable at the top of this notebook.

In [ ]:
import pandas as pd

contests = client.list_contests()
contests_df = pd.DataFrame(contests)
contests_df[["contest_id", "name", "status", "twin_id", "start_date", "end_date"]]

Copy a `contest_id` from the table above and paste it here.

In [ ]:
CONTEST_ID = "replace-with-contest-id"  # copy a contest_id from the table above

## 3. Register for the Contest

Before you can join a contest stream or submit predictions, your participant account must be registered for that contest. Calling `register()` is safe to repeat: if you are already registered, the SDK returns a confirmation instead of raising an error.

In [ ]:
client.register(CONTEST_ID)

## 4. Collect Live Observations

`collect()` listens to the contest stream for a fixed duration and returns a list of observations. Each observation includes a `sequence_id`, a `timestamp`, and a `sensors` dictionary containing the measurements published by the contest.

In [ ]:
observations = await client.collect(CONTEST_ID, duration_seconds=10, csv_path="epic_observations.csv")
len(observations), observations[:2]

## 5. Inspect the Dataset

For analysis, flatten the `sensors` dictionary into columns and load the result into a pandas DataFrame. This makes it easier to inspect sensor values, compute features, and build predictions.

In [ ]:
import pandas as pd

rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp": obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df.head()

## 6. Plot a Sensor Channel

Plotting a channel helps you understand the signal scale and whether there are visible changes or anomalies. This example plots the first sensor column available in the dataset.

In [ ]:
import matplotlib.pyplot as plt

sensor_columns = [c for c in df.columns if c not in {"sequence_id", "timestamp"}]
sensor_name = sensor_columns[0]

df.plot(x="sequence_id", y=sensor_name, figsize=(10, 4), title=f"{sensor_name} over time")
plt.show()

## 7. Build a Trivial Prediction

A real solution should train a model or use domain knowledge. For a first submission, use a simple baseline: predict that future values stay equal to the latest observed value. This is not competitive, but it verifies the submission workflow.

In [ ]:
latest = df.iloc[-1]
prediction_from_sequence = int(latest["sequence_id"])

payload = {
    "forecast": {
        "horizon_1": {sensor_name: float(latest[sensor_name])},
        "horizon_5": {sensor_name: float(latest[sensor_name])},
        "horizon_10": {sensor_name: float(latest[sensor_name])},
    }
}
payload

## 8. Submit the Prediction

Submit the prediction to the contest. The `prediction_from_sequence` anchors your forecast to a published observation, which is required for submission integrity.

In [ ]:
submission = client.submit(
    contest_id=CONTEST_ID,
    task_id="forecasting",
    prediction_from_sequence=prediction_from_sequence,
    payload=payload,
)
submission

## 9. Check Scores and Leaderboard

Scoring may run asynchronously, so a new submission can remain pending briefly. These calls show your scored submissions and the current contest leaderboard.

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

scores, leaderboard